In [ ]:
# ============================================
# Imports
# ============================================
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.linalg import expm
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from scipy.sparse.linalg import expm_multiply
from scipy.sparse import lil_matrix, csr_matrix



# ============================================
# BASIS CONSTRUCTION
# ============================================
def build_basis(inv_value, photon_cutoff, phonon_cutoff):
    """
    Construct basis of states (n_s, n_as, n_ph) satisfying:
        n_as + n_ph - n_s = inv_value

    This corresponds to the conserved quantity of the Hamiltonian.
    """
    basis = []
    for n_s in range(photon_cutoff):
        for n_as in range(photon_cutoff):
            for n_ph in range(phonon_cutoff):
                if n_as + n_ph - n_s == inv_value:
                    basis.append((n_s, n_as, n_ph))
    return basis


# ============================================
# OPERATOR CONSTRUCTION
# ============================================
def build_operator(basis, *, oprtr):
    """
    Construct matrix representation of operators in the restricted basis.

    Supported operators:
        aS^+ b^+   (Stokes creation)
        aS b       (Stokes annihilation)
        aaS^+ b    (anti-Stokes creation)
        aaS b^+    (anti-Stokes annihilation)
        nS         (Stokes number)
        naS        (anti-Stokes number)
    """
    dim = len(basis)
    matrix = np.zeros((dim, dim), dtype=complex)
    basis_index = {state: i for i, state in enumerate(basis)}

    for j, (n_s, n_as, n_ph) in enumerate(basis):

        if oprtr == 'aS^+ b^+':
            new_state = (n_s + 1, n_as, n_ph + 1)
            if new_state in basis_index:
                i = basis_index[new_state]
                matrix[i, j] = np.sqrt(n_s + 1) * np.sqrt(n_ph + 1)

        elif oprtr == 'aS b':
            if n_s > 0 and n_ph > 0:
                new_state = (n_s - 1, n_as, n_ph - 1)
                if new_state in basis_index:
                    i = basis_index[new_state]
                    matrix[i, j] = np.sqrt(n_s) * np.sqrt(n_ph)

        elif oprtr == 'aaS^+ b':
            if n_ph > 0:
                new_state = (n_s, n_as + 1, n_ph - 1)
                if new_state in basis_index:
                    i = basis_index[new_state]
                    matrix[i, j] = np.sqrt(n_as + 1) * np.sqrt(n_ph)

        elif oprtr == 'aaS b^+':
            if n_as > 0:
                new_state = (n_s, n_as - 1, n_ph + 1)
                if new_state in basis_index:
                    i = basis_index[new_state]
                    matrix[i, j] = np.sqrt(n_as) * np.sqrt(n_ph + 1)

        elif oprtr == 'nS':
            matrix[j, j] = n_s

        elif oprtr == 'naS':
            matrix[j, j] = n_as

        else:
            raise ValueError("Unknown operator")

    return matrix


# ============================================
# NRF COMPUTATION
# ============================================
def compute_NRF(g_s, g_as, nbar):
    """
    Compute Noise Reduction Factor (NRF):

        NRF = Var(n_S - n_aS) / <n_S + n_aS>

    averaged over thermal phonon distribution.
    """
    coeffs = np.diag(qt.thermal_dm(phonon_cutoff, nbar).full())
    coeffs = coeffs / np.sum(coeffs)  # ensure normalization

    nS_avg = 0.0
    naS_avg = 0.0
    diff_sqr_avg = 0.0
    diff_avg = 0.0
    sum_avg = 0.0

    for n_ph, coeff in enumerate(coeffs):

        # skip negligible thermal weights
        if coeff < 1e-14:
            continue

        # build invariant subspace
        basis = build_basis(n_ph, photon_cutoff, phonon_cutoff)
        basis_index = {state: i for i, state in enumerate(basis)}

        # initial state |0,0,n_ph>
        if (0, 0, n_ph) not in basis_index:
            continue

        i0 = basis_index[(0, 0, n_ph)]

        # Hamiltonian construction
        stokes = build_operator(basis, oprtr='aS^+ b^+')
        stokes_dag = build_operator(basis, oprtr='aS b')
        anti = build_operator(basis, oprtr='aaS^+ b')
        anti_dag = build_operator(basis, oprtr='aaS b^+')

        H = (g_s * stokes + np.conjugate(g_s) * stokes_dag +
             g_as * anti + np.conjugate(g_as) * anti_dag)

        # observables
        nS = build_operator(basis, oprtr='nS')
        naS = build_operator(basis, oprtr='naS')

        diff = nS - naS
        diff_sqr = diff @ diff
        summ = nS + naS

        # # unitary evolution
        # U = expm(-1.0j * H)
        # Udag = U.conjugate().T

        # # accumulate expectation values
        # nS_avg += coeff * (Udag @ nS @ U)[i0, i0].real
        # naS_avg += coeff * (Udag @ naS @ U)[i0, i0].real
        # diff_avg += coeff * (Udag @ diff @ U)[i0, i0].real
        # diff_sqr_avg += coeff * (Udag @ diff_sqr @ U)[i0, i0].real
        # sum_avg += coeff * (Udag @ summ @ U)[i0, i0].real

        
        # psi_i = np.zeros(H.shape[0], dtype=complex)
        # psi_i[i0] = 1.0
        # psi_f = expm_multiply(-1.0j * H, psi_i)
        
        # nS_avg += coeff * np.vdot(psi_f, nS @ psi_f).real
        # naS_avg += coeff * np.vdot(psi_f, naS @ psi_f).real
        # diff_avg += coeff * np.vdot(psi_f, diff @ psi_f).real
        # diff_sqr_avg += coeff * np.vdot(psi_f, diff_sqr @ psi_f).real
        # sum_avg += coeff * np.vdot(psi_f, summ @ psi_f).real
    
        psi_i = np.zeros(H.shape[0], dtype=complex)
        psi_i[i0] = 1.0
        H_sparse = csr_matrix(H, dtype=complex)
        psi_f = expm_multiply(-1.0j * H_sparse, psi_i)
         
        nS_avg += coeff * np.vdot(psi_f, nS @ psi_f).real
        naS_avg += coeff * np.vdot(psi_f, naS @ psi_f).real
        diff_avg += coeff * np.vdot(psi_f, diff @ psi_f).real
        diff_sqr_avg += coeff * np.vdot(psi_f, diff_sqr @ psi_f).real
        sum_avg += coeff * np.vdot(psi_f, summ @ psi_f).real
        
    if sum_avg < 1e-12:
        return np.nan

    return (diff_sqr_avg - diff_avg**2) / sum_avg


def compute_probabilities(g_s, g_as, nbar):
    """
    Compute photon number distributions:

        P_nS[n]  = Prob(n_S = n)
        P_naS[n] = Prob(n_aS = n)

    averaged over thermal phonon distribution.
    """

    # Thermal phonon distribution
    coeffs = np.real(np.diag(qt.thermal_dm(phonon_cutoff, nbar).full()))
    coeffs = coeffs / np.sum(coeffs)  # ✅ IMPORTANT: normalize!

    # Output distributions
    P_nS = np.zeros(photon_cutoff)
    P_naS = np.zeros(photon_cutoff)

    for n_ph, coeff in enumerate(coeffs):

        if coeff < 1e-14:
            continue

        # Build invariant subspace
        basis = build_basis(n_ph, photon_cutoff, phonon_cutoff)
        basis_index = {state: i for i, state in enumerate(basis)}

        # Initial state |0,0,n_ph>
        if (0, 0, n_ph) not in basis_index:
            continue

        i0 = basis_index[(0, 0, n_ph)]

        # Build Hamiltonian
        stokes = build_operator(basis, oprtr='aS^+ b^+')
        stokes_dag = build_operator(basis, oprtr='aS b')

        anti = build_operator(basis, oprtr='aaS^+ b')
        anti_dag = build_operator(basis, oprtr='aaS b^+')

        H = (g_s * stokes + np.conjugate(g_s) * stokes_dag +
             g_as * anti + np.conjugate(g_as) * anti_dag)

        # Time evolution
        U = expm(-1.0j * H)

        # Initial state vector
        psi_i = np.zeros(len(basis), dtype=complex)
        psi_i[i0] = 1.0

        # Final state
        psi_f = U @ psi_i

        # Probabilities in basis
        probs = np.abs(psi_f)**2

        # Marginalize distributions
        for idx, (n_s, n_as, _) in enumerate(basis):
            p = probs[idx]
            if abs(p.imag) > 1.0e-12:
                raise ValueError(f'{p = }')
            P_nS[n_s] += coeff * float(p.real)
            P_naS[n_as] += coeff * float(p.real)

    return P_nS, P_naS


def compute_joint_probabilities(g_s, g_as, nbar):
    """
    Compute joint photon number distribution:

        P_joint[n_S, n_aS] = Prob(n_S, n_aS)

    averaged over thermal phonon distribution.
    """

    # Thermal phonon distribution
    coeffs = np.real(np.diag(qt.thermal_dm(phonon_cutoff, nbar).full()))
    coeffs = coeffs / np.sum(coeffs)  # normalize

    # Joint distribution
    P_joint = np.zeros((photon_cutoff, photon_cutoff))

    for n_ph, coeff in enumerate(coeffs):
        if coeff < 1e-14:
            continue
        
        # Build invariant subspace
        basis = build_basis(n_ph, photon_cutoff, phonon_cutoff)
        basis_index = {state: i for i, state in enumerate(basis)}

        # Initial state |0,0,n_ph>
        if (0, 0, n_ph) not in basis_index:
            continue

        i0 = basis_index[(0, 0, n_ph)]

        # Build Hamiltonian
        stokes = build_operator(basis, oprtr='aS^+ b^+')
        stokes_dag = build_operator(basis, oprtr='aS b')

        anti = build_operator(basis, oprtr='aaS^+ b')
        anti_dag = build_operator(basis, oprtr='aaS b^+')

        H = (g_s * stokes + np.conjugate(g_s) * stokes_dag +
             g_as * anti + np.conjugate(g_as) * anti_dag)

        # Time evolution
        U = expm(-1.0j * H)

        # Initial state
        psi_i = np.zeros(len(basis), dtype=complex)
        psi_i[i0] = 1.0

        # Final state
        psi_f = U @ psi_i

        # Probabilities
        probs = np.abs(psi_f)**2

        # Fill joint distribution
        for idx, (n_s, n_as, _) in enumerate(basis):
            p = probs[idx]
            if abs(p.imag) > 1e-12:
                raise ValueError(f'{p = }')
            P_joint[n_s, n_as] += coeff * float(p.real)
    return P_joint


# ============================================
# PARALLEL WORKER
# ============================================
def worker(params):
    """Wrapper for multiprocessing"""
    i, j, k, nbar, g_s, g_as = params
    val = compute_NRF(g_s, g_as, nbar)
    return (i, j, k, val)


# ============================================
# MAIN SCRIPT (IMPORTANT FOR MULTIPROCESSING)
# ============================================
photon_cutoff = 10
phonon_cutoff = 10

# nbar_array = np.array([0.01, 0.05, 0.1])
nbar_array = np.array([0.1])
g_s_array =  np.linspace(0.001, 0.2, 6)
g_as_array = np.linspace(0.001, 0.2, 7)
print(f'{g_s_array = }')

NRF = np.zeros((len(nbar_array), len(g_s_array), len(g_as_array)))

# prepare parameter grid
tasks = []
for i, nbar in enumerate(nbar_array):
    for j, g_s in enumerate(g_s_array):
        for k, g_as in enumerate(g_as_array):
            tasks.append((i, j, k, nbar, g_s, g_as))

# parallel computation with progress bar
# with Pool(cpu_count()) as pool:
#     results = list(
#         tqdm(
#             pool.imap(worker, tasks),
#             total=len(tasks),
#             desc="Computing NRF"
#         )
#     )

# sequential version
results = list(map(worker, tasks))

# fill result array
for i, j, k, val in results:
    if abs(float(val.imag)) > 1.0e-12:
        raise ValueError(f'complex value of NRF[{i, j, k}]: {val}')
    NRF[i, j, k] = float(val.real)

# ============================================
# VISUALIZATION
# ============================================
for idx, nbar in enumerate(nbar_array):
    plt.figure()
    plt.contourf(g_s_array, g_as_array, NRF[idx].T, levels=100)
    plt.colorbar()
    plt.title(f"NRF (nbar={nbar})")
    plt.xlabel("g_s")
    plt.ylabel("g_as")
    plt.show()

In [ ]:
# select parameters for probability calculation

nbar_ind, g_s_ind, g_as_ind = 0, 3, 4

nbar = nbar_array[nbar_ind]
g_s = g_s_array[g_s_ind]
g_as = g_as_array[g_as_ind]
print(f'{nbar = }')
print(f'{g_s = }')
print(f'{g_as = }')

print(f'{NRF[nbar_ind, g_s_ind, g_as_ind] = }')
P_nS, P_naS = compute_probabilities(g_s, g_as, nbar)

P_joint = compute_joint_probabilities(g_s, g_as, nbar)

plt.contourf(np.arange(photon_cutoff), np.arange(photon_cutoff), P_joint, levels=100)
plt.show()

In [ ]:
# ============================================
# SELECT PARAMETERS FOR SANITY CHECK
# ============================================
nbar_ind, g_s_ind, g_as_ind = 0, 2, 5

nbar = nbar_array[nbar_ind]
g_s = g_s_array[g_s_ind]
g_as = g_as_array[g_as_ind]
print(f'{nbar = }')
print(f'{g_s = }')
print(f'{g_as = }')


# ============================================
# INITIAL STATE: |0>_S ⊗ |0>_aS ⊗ thermal_phonons
# ============================================
rho_init = qt.tensor(
    qt.fock_dm(photon_cutoff, 0),          # Stokes mode vacuum
    qt.fock_dm(photon_cutoff, 0),          # anti-Stokes mode vacuum
    qt.thermal_dm(phonon_cutoff, nbar)     # phonon thermal state
)


# ============================================
# HAMILTONIAN CONSTRUCTION
# H = g_s a_S^† b^† + g_as a_aS^† b + h.c.
# ============================================
aS_create = qt.tensor(
    qt.create(photon_cutoff),
    qt.qeye(photon_cutoff),
    qt.create(phonon_cutoff)
)

aAS_create = qt.tensor(
    qt.qeye(photon_cutoff),
    qt.create(photon_cutoff),
    qt.destroy(phonon_cutoff)
)

H_qt = g_s * aS_create + g_as * aAS_create
H_qt = H_qt + H_qt.dag()   # make Hermitian


# ============================================
# TIME EVOLUTION (t = 1)
# ============================================
result = qt.mesolve(H_qt, rho_init, tlist=[0.0, 1.0])
rho_f = result.states[-1]


# ============================================
# NUMBER OPERATORS
# ============================================
nS_op = qt.tensor(
    qt.num(photon_cutoff),
    qt.qeye(photon_cutoff),
    qt.qeye(phonon_cutoff)
)

naS_op = qt.tensor(
    qt.qeye(photon_cutoff),
    qt.num(photon_cutoff),
    qt.qeye(phonon_cutoff)
)


# ============================================
# EXPECTATION VALUES
# ============================================
nS_avg_qt = qt.expect(nS_op, rho_f)
naS_avg_qt = qt.expect(naS_op, rho_f)

print(f"{nS_avg_qt = }")
print(f"{naS_avg_qt = }")


# ============================================
# NRF COMPUTATION
# ============================================
variance_diff = qt.variance(nS_op - naS_op, rho_f)
mean_sum = qt.expect(nS_op + naS_op, rho_f)

NRF_qt = variance_diff / mean_sum

print(f"{variance_diff = }")
print(f"{mean_sum = }")
print(f"                                                   {NRF_qt = }")
print(f"(manual code: {NRF[nbar_ind, g_s_ind, g_as_ind] = })")


# ============================================
# PROBABILITY SANITY CHECK
# ============================================
P_nS, P_naS = compute_probabilities(g_s, g_as, nbar)


# ---- Stokes mode ----
print("\n--- Stokes mode ---")
print(f"{P_nS.sum() = }")

mean_nS_manual = np.sum(P_nS * np.arange(len(P_nS)))
var_nS_manual = np.sum(P_nS * np.arange(len(P_nS))**2) - mean_nS_manual**2

print(f" QuTiP mean = {qt.expect(nS_op, rho_f)}")
print(f"Manual mean = {mean_nS_manual}")

print(f"nQuTiP variance = {qt.variance(nS_op, rho_f)}")
print(f"Manual variance = {var_nS_manual}")


# ---- anti-Stokes mode ----
print("\n--- anti-Stokes mode ---")
print(f"{P_naS.sum() = }")

mean_naS_manual = np.sum(P_naS * np.arange(len(P_naS)))
var_naS_manual = np.sum(P_naS * np.arange(len(P_naS))**2) - mean_naS_manual**2

print(f" QuTiP mean = {qt.expect(naS_op, rho_f)}")
print(f"Manual mean = {mean_naS_manual}")

print(f" QuTiP variance = {qt.variance(naS_op, rho_f)}")
print(f"Manual variance = {var_naS_manual}")

In [ ]:
np.savez_compressed('./NRF_and_probs_data.npz', 
                    gS=g_s_array,
                    gaS=g_as_array,
                    nbar=nbar_array,
                    pnS=P_nS,
                    pnaS=P_naS,
                    pjoint=P_joint,
                    NRF=NRF,
                    cutoffs=np.array([photon_cutoff, phonon_cutoff])
                   )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
# Axes3D import is required for 3D plotting, even if not used explicitly


# =========================
# Load precomputed data
# =========================
# The .npz file contains arrays computed previously
data = np.load('./NRF_and_probs_data.npz')  


# Extract arrays from file
# gS, gaS: parameter grids (coupling strengths or similar)
# pnS, pnaS: marginal photon number distributions
# pjoint: joint photon number distribution
# NRF: noise reduction factor
g_s_array = data['gS']
g_as_array = data['gaS']
P_nS = data['pnS']
P_naS = data['pnaS']
P_joint = data['pjoint']
NRF = data['NRF']
photon_cutoff, phonon_cutoff = data['cutoffs']

# If NRF has an extra dimension (e.g., time or index), select the first slice
NRF = NRF[0, ...]

# Print minimum value for quick diagnostics
print(f'{NRF.min() = }')
print(f'{NRF.max() = }')


# =========================
# 1) NRF contour plot
# =========================
fig, ax = plt.subplots(figsize=(5, 4))

# Filled contour plot of NRF over parameter space (g_S, g_aS)
# Transpose is used to match axis orientation
cf = ax.contourf(g_s_array, g_as_array, NRF.T, levels=500, cmap='jet')

# Add colorbar with selected tick marks
fig.colorbar(cf, ax=ax, ticks=[2, 4, 6, 8, 10])

# Compute contour levels based on quantiles (10%, 20%, 30%)
# This highlights regions of relatively low NRF
level_vals = [
    np.round(np.quantile(NRF.ravel(), 0.1), 4), 
    np.round(np.quantile(NRF.ravel(), 0.2), 4), 
    np.round(np.quantile(NRF.ravel(), 0.3), 4)
]
print(f'{level_vals = }')

# Overlay contour lines at selected quantile levels
cs = ax.contour(
    g_s_array, g_as_array, NRF.T,
    levels=level_vals,
    colors='white',
    linewidths=2,
    linestyles='--'
)

# Optional: label contour lines (currently disabled)
# ax.clabel(cs, fmt=f'{level_val:.2f}', colors='white')

# Axis labels and title (LaTeX formatting)
ax.set_title(r'$NRF(g_S, g_{aS})$')
ax.set_xlabel(r'$g_S$')
ax.set_ylabel(r'$g_{aS}$')
ax.set_xticks([0.0, 0.5, 1.0, 1.5])
ax.set_yticks([0.0, 0.5, 1.0, 1.5])

# Adjust layout and save figure
plt.tight_layout()
plt.savefig(f'./NRF.pdf', dpi=400)
plt.show()
plt.close(fig)


# =========================
# 2) Stokes photon number distribution
# =========================
fig, ax = plt.subplots(figsize=(4, 3))

# Plot first few probabilities (truncate to 5 values for clarity)
ax.bar(np.arange(len(P_nS))[:5], P_nS[:5], color='crimson')

ax.set_title('Stokes photon statistics')
ax.set_xlabel(r'$n_S$')   # photon number
ax.set_ylabel(r'$P_S$')   # probability
ax.set_xticks([0, 2, 4])
ax.set_yticks([0.0, 0.4, 0.8])

plt.tight_layout()
plt.savefig(f'./prob_S.pdf', dpi=400)
plt.show()
plt.close(fig)


# =========================
# 3) anti-Stokes photon number distribution
# =========================
fig, ax = plt.subplots(figsize=(4, 3))

# Same as above but for anti-Stokes photons
ax.bar(np.arange(len(P_naS))[:5], P_naS[:5], color='dodgerblue')

ax.set_title('anti-Stokes photon statistics')
ax.set_xlabel(r'$n_{aS}$')
ax.set_ylabel(r'$P_{aS}$')
ax.set_xticks([0, 2, 4])
ax.set_yticks([0.0, 0.4, 0.8])

plt.tight_layout()
plt.savefig(f'./prob_aS.pdf', dpi=400)
plt.show()
plt.close(fig)


# =========================
# 4) Joint photon distribution (3D bar plot)
# =========================
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection='3d')

# Optional aesthetic tweaks (currently disabled)
# ax.xaxis.pane.fill = False
# ax.yaxis.pane.fill = False
# ax.xaxis.pane.set_edgecolor('none')
# ax.yaxis.pane.set_edgecolor('none')

# Define discrete photon number grid (shifted for bar centering)
x = np.arange(4) - 0.25
y = np.arange(4) - 0.25

# Create 2D grid of positions
xpos, ypos = np.meshgrid(x, y, indexing='ij')

# Flatten grid for bar3d input
xpos = xpos.ravel()
ypos = ypos.ravel()

# Base (z=0) for all bars
zpos = np.zeros_like(xpos)

# Bar dimensions (constant width in x and y)
dx = dy = 0.5 * np.ones_like(zpos)

# Heights correspond to joint probability values (truncated to 6x6 block)
dz = P_joint[:4, :4].ravel()

# Create 3D bar plot
ax.bar3d(xpos, ypos, zpos, dx, dy, dz, color='darkmagenta')

# Labels and title
ax.set_xlabel(r'$n_S$')
ax.set_ylabel(r'$n_{aS}$')
ax.set_zlabel(r'$P$')
ax.set_title(r'$P(n_S, n_{aS})$')

# Set ticks for better readability
ax.set_xticks([0, 2, 4])
ax.set_yticks([0, 2, 4])
ax.set_zticks([0, 0.4, 0.8])

plt.tight_layout()
plt.savefig(f'./prob_joint.pdf', dpi=400)
plt.show()
plt.close(fig)

In [ ]:
# comparison with analytical result

import numpy as np
import qutip as qt

photon_cutoff = 10
phonon_cutoff = 9

aS_dag = qt.create(photon_cutoff)
aS = qt.destroy(photon_cutoff)

aaS_dag = qt.create(photon_cutoff)
aaS = qt.destroy(photon_cutoff)

b_dag = qt.create(phonon_cutoff)
b = qt.destroy(phonon_cutoff)


nbar = 0.1
rho_init = qt.tensor(qt.fock_dm(photon_cutoff, 0), 
                     qt.fock_dm(photon_cutoff, 0), 
                     qt.thermal_dm(phonon_cutoff, nbar)
                     )
# rho_init
g_S = 0.1
g_aS = 0.2
H = g_S * qt.tensor(aS_dag, qt.qeye(photon_cutoff), b_dag) + g_aS * qt.tensor(qt.qeye(photon_cutoff), aaS_dag, b)
H = H + H.dag()

result = qt.mesolve(H, rho_init, tlist=[0.0, 1.0])

rho_final = result.states[-1]

# -----------------------------------
# Trace out phonon subsystem
# keep subsystem 0 (Stokes photon) and 1 (anti-Stokes photon)
# -----------------------------------
rho_photons = rho_final.ptrace([0, 1])
rho_photons


In [ ]:
analytical_coeffs = {
    '00': 1.0 - abs(g_S) ** 2 * (nbar + 1) - abs(g_aS) ** 2 * nbar,
    '10': abs(g_S) ** 2 * (nbar + 1),
    '01': abs(g_aS) ** 2 * nbar,
    '1->0': g_S * g_aS * nbar,
    '0->1': g_S.conjugate() * g_aS.conjugate() * nbar
}

numerical_coeffs = {
    '00': <extract corresponding element from rho_final> (),
    ...
}

for key in analytical_coeffs.keys():
    print(key)
    print(analytical_coeffs[key])
    print(numerical_coeffs[key])
    print()

In [ ]:
import numpy as np
import qutip as qt

photon_cutoff = 3
phonon_cutoff = 5

aS_dag = qt.create(photon_cutoff)
aS = qt.destroy(photon_cutoff)

aaS_dag = qt.create(photon_cutoff)
aaS = qt.destroy(photon_cutoff)

b_dag = qt.create(phonon_cutoff)
b = qt.destroy(phonon_cutoff)

nbar = 0.1
rho_init = qt.tensor(
    qt.fock_dm(photon_cutoff, 0),
    qt.fock_dm(photon_cutoff, 0),
    qt.thermal_dm(phonon_cutoff, nbar)
)

g_S = 0.1
g_aS = 0.2

H = (
    g_S * qt.tensor(aS_dag, qt.qeye(photon_cutoff), b_dag)
    + g_aS * qt.tensor(qt.qeye(photon_cutoff), aaS_dag, b)
)
H = H + H.dag()

# evolve to time t = 1
t_final = 1.0
result = qt.mesolve(H, rho_init, tlist=[0.0, t_final])

rho_final = result.states[-1]

# trace out phonons: keep only Stokes and anti-Stokes photon modes
rho_photons = rho_final.ptrace([0, 1])

# convert to dense matrix
rho_ph = rho_photons.full()

rho_photons

In [ ]:
# helper: index of |nS, n_aS> in the reduced 2-mode Hilbert space
def idx(nS, naS, cutoff):
    return nS * cutoff + naS

i00 = idx(0, 0, photon_cutoff)   # |0,0>
i10 = idx(1, 0, photon_cutoff)   # |1,0>
i01 = idx(0, 1, photon_cutoff)   # |0,1>
i11 = idx(1, 1, photon_cutoff)   # |1,1>

# second-order analytical coefficients
# since t_final = 1, these are exactly the formulas you wrote;
# for general t, multiply all quadratic terms by t_final**2
analytical_coeffs = {
    '00': 1.0 - abs(g_S)**2 * (nbar + 1) - abs(g_aS)**2 * nbar,
    '10': abs(g_S)**2 * (nbar + 1),
    '01': abs(g_aS)**2 * nbar,
    '11<->00': g_S * g_aS * nbar,
    '00<->11': np.conjugate(g_S * g_aS * nbar),
}

# numerical coefficients extracted from reduced density matrix
numerical_coeffs = {
    '00': rho_ph[i00, i00],   # <0,0| rho |0,0>
    '10': rho_ph[i10, i10],   # <1,0| rho |1,0>
    '01': rho_ph[i01, i01],   # <0,1| rho |0,1>
    '11<->00': rho_ph[i11, i00],  # <1,1| rho |0,0>
    '00<->11': rho_ph[i00, i11],  # <0,0| rho |1,1>
}

for key in analytical_coeffs.keys():
    print(key)
    print("analytical =", analytical_coeffs[key])
    print("numerical  =", numerical_coeffs[key])
    print("difference =", numerical_coeffs[key] - analytical_coeffs[key])
    print()

In [ ]:
import numpy as np
import qutip as qt
from itertools import product

# ============================================================
# Settings
# ============================================================
photon_cutoff = 6
phonon_cutoff = 14
t_final = 1.0

# Fixed phonon thermal occupation
nbar = 0.15

# Small parameter grids for fitting
gS_values = np.array([-0.06, -0.03, 0.00, 0.03, 0.06], dtype=float)
gaS_values = np.array([-0.06, -0.03, 0.00, 0.03, 0.06], dtype=float)

gS_values = np.linspace(-0.06, +0.06, 10)
gaS_values = np.linspace(-0.06, +0.06, 10)
# Threshold for printing tiny coefficients
print_tol = 1e-8

# ============================================================
# Basis utilities
# ============================================================
def idx(nS, naS, cutoff):
    """Index of |nS, naS> in the 2-mode photon Hilbert space."""
    return nS * cutoff + naS

block_states = [(0, 0), (1, 0), (0, 1), (1, 1)]

# ============================================================
# Polynomial basis in (gS, gaS), degree <= 2
# ============================================================
monomial_names = [
    "1",
    "gS",
    "gaS",
    "gS^2",
    "gaS^2",
    "gS*gaS",
]

def feature_vector(gS, gaS):
    return np.array([
        1.0,
        gS,
        gaS,
        gS**2,
        gaS**2,
        gS * gaS,
    ], dtype=float)

# ============================================================
# QuTiP operators independent of parameters
# ============================================================
aS_dag = qt.create(photon_cutoff)
aS = qt.destroy(photon_cutoff)

aaS_dag = qt.create(photon_cutoff)
aaS = qt.destroy(photon_cutoff)

b_dag = qt.create(phonon_cutoff)
b = qt.destroy(phonon_cutoff)

Iph = qt.qeye(photon_cutoff)
Iphn = qt.qeye(phonon_cutoff)

HS_term = qt.tensor(aS_dag, Iph, b_dag)
HaS_term = qt.tensor(Iph, aaS_dag, b)

# ============================================================
# Build dataset
# ============================================================
X_rows = []
Y = {key: [] for key in product(block_states, block_states)}

for gS, gaS in product(gS_values, gaS_values):
    rho_init = qt.tensor(
        qt.fock_dm(photon_cutoff, 0),
        qt.fock_dm(photon_cutoff, 0),
        qt.thermal_dm(phonon_cutoff, nbar)
    )

    H = gS * HS_term + gaS * HaS_term
    H = H + H.dag()

    result = qt.mesolve(H, rho_init, tlist=[0.0, t_final])
    rho_final = result.states[-1]

    # trace out phonons
    rho_ph = rho_final.ptrace([0, 1]).full()

    X_rows.append(feature_vector(gS, gaS))

    for (a, b), (c, d) in product(block_states, block_states):
        i = idx(a, b, photon_cutoff)
        j = idx(c, d, photon_cutoff)
        Y[((a, b), (c, d))].append(rho_ph[i, j])

X = np.vstack(X_rows)

# ============================================================
# Fit each block element
# ============================================================
def fit_element(y_complex):
    y_complex = np.asarray(y_complex)
    coeffs_re, *_ = np.linalg.lstsq(X, y_complex.real, rcond=None)
    coeffs_im, *_ = np.linalg.lstsq(X, y_complex.imag, rcond=None)
    return coeffs_re + 1j * coeffs_im

fitted_coeffs = {}
for key in Y:
    fitted_coeffs[key] = fit_element(Y[key])

# ============================================================
# Pretty-printing
# ============================================================
def format_complex(z, tol=print_tol):
    re = z.real
    im = z.imag

    re_small = abs(re) < tol
    im_small = abs(im) < tol

    if re_small and im_small:
        return "0"
    if im_small:
        return f"{re:.8g}"
    if re_small:
        return f"{im:.8g}j"

    sign = "+" if im >= 0 else "-"
    return f"({re:.8g} {sign} {abs(im):.8g}j)"

def polynomial_to_string(coeffs, names, tol=print_tol):
    terms = []
    for c, name in zip(coeffs, names):
        if abs(c) < tol:
            continue
        if name == "1":
            terms.append(f"{format_complex(c, tol)}")
        else:
            terms.append(f"{format_complex(c, tol)} * {name}")
    return " + ".join(terms) if terms else "0"

# ============================================================
# Print all fitted block elements
# ============================================================
# print("=" * 80)
# print(f"Fitted photon-density-matrix block for fixed nbar = {nbar}")
# print("Polynomial basis: 1, gS, gaS, gS^2, gaS^2, gS*gaS")
# print("=" * 80)
# print()

# for (a, b), (c, d) in product(block_states, block_states):
#     coeffs = fitted_coeffs[((a, b), (c, d))]
#     expr = polynomial_to_string(coeffs, monomial_names)
#     print(f"rho_({a}{b},{c}{d}) = {expr}")
#     print()

# ============================================================
# Optional: print only non-negligible matrix elements
# ============================================================
print("=" * 80)
print("Only non-negligible fitted elements")
print("=" * 80)
print()

for (a, b), (c, d) in product(block_states, block_states):
    coeffs = fitted_coeffs[((a, b), (c, d))]
    if np.max(np.abs(coeffs)) > print_tol:
        expr = polynomial_to_string(coeffs, monomial_names)
        print(f"rho_({a}{b},{c}{d}) = {expr}")

In [ ]:
import numpy as np
import qutip as qt
from itertools import product


def get_NRF_using_qutip(g_S, g_aS, nbar, photon_cutoff, phonon_cutoff, t_final=1.0):
    aS_dag = qt.create(photon_cutoff)
    aS = qt.destroy(photon_cutoff)

    aaS_dag = qt.create(photon_cutoff)
    aaS = qt.destroy(photon_cutoff)

    b_dag = qt.create(phonon_cutoff)
    b = qt.destroy(phonon_cutoff)

    Iph = qt.qeye(photon_cutoff)
    Iphn = qt.qeye(phonon_cutoff)

    rho_init = qt.tensor(
        qt.fock_dm(photon_cutoff, 0),
        qt.fock_dm(photon_cutoff, 0),
        qt.thermal_dm(phonon_cutoff, nbar)
    )

    H = (
        g_S * qt.tensor(aS_dag, Iph, b_dag)
        + g_aS * qt.tensor(Iph, aaS_dag, b)
    )
    H = H + H.dag()

    result = qt.mesolve(H, rho_init, tlist=[0.0, t_final])
    rho_final = result.states[-1]

    rho_ph = rho_final.ptrace([0, 1])

    nS_op = qt.tensor(aS_dag * aS, Iph)
    naS_op = qt.tensor(Iph, aaS_dag * aaS)

    diff_op = nS_op - naS_op
    sum_op = nS_op + naS_op

    nrf_num = qt.variance(diff_op, rho_ph)
    nrf_den = qt.expect(sum_op, rho_ph)

    if abs(nrf_den) < 1e-14:
        return np.nan

    return float(np.real_if_close(nrf_num / nrf_den))


def fit_NRF_polynomial_fixed_nbar(
    nbar=0.1,
    photon_cutoff=12,
    phonon_cutoff=12,
    t_final=1.0,
    gS_values=None,
    gaS_values=None,
    print_tol=1e-8
):
    if gS_values is None:
        gS_values = np.array([-0.05, -0.025, -0.0125, 0.0125, 0.025, 0.05], dtype=float)
    if gaS_values is None:
        gaS_values = np.array([-0.05, -0.025, -0.0125, 0.0125, 0.025, 0.05], dtype=float)

    # Polynomial basis up to degree 2:
    # F(g_S, g_aS) = c0 + c1*g_S + c2*g_aS + c3*g_S^2 + c4*g_aS^2 + c5*g_S*g_aS
    monomial_names = ["1", "g_S", "g_aS", "g_S^2", "g_aS^2", "g_S*g_aS"]

    def feature_vector(g_S, g_aS):
        return np.array([
            1.0,
            g_S,
            g_aS,
            g_S**2,
            g_aS**2,
            g_S * g_aS
        ], dtype=float)

    X_rows = []
    y_vals = []

    for g_S, g_aS in product(gS_values, gaS_values):
        nrf = get_NRF_using_qutip(
            g_S=g_S,
            g_aS=g_aS,
            nbar=nbar,
            photon_cutoff=photon_cutoff,
            phonon_cutoff=phonon_cutoff,
            t_final=t_final
        )

        if np.isnan(nrf):
            continue

        X_rows.append(feature_vector(g_S, g_aS))
        y_vals.append(nrf)

    X = np.vstack(X_rows)
    y = np.array(y_vals, dtype=float)

    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)

    def polynomial_to_string(coeffs, names, tol=print_tol):
        terms = []
        for c, name in zip(coeffs, names):
            if abs(c) < tol:
                continue
            if name == "1":
                terms.append(f"{c:.10g}")
            else:
                terms.append(f"{c:.10g} * {name}")
        return " + ".join(terms) if terms else "0"

    poly_str = polynomial_to_string(coeffs, monomial_names, tol=print_tol)

    print(f"Fixed nbar = {nbar}")
    print("Fitted polynomial:")
    print(f"NRF(g_S, g_aS, nbar={nbar}) = {poly_str}")
    print()

    print("Coefficients:")
    for name, c in zip(monomial_names, coeffs):
        print(f"{name:>8s} : {c:.12e}")

    return {
        "nbar": nbar,
        "monomial_names": monomial_names,
        "coeffs": coeffs,
        "polynomial_string": poly_str,
        "X": X,
        "y": y
    }


# =========================
# Example of use
# =========================

nbar_arr = np.linspace(0.1, 0.5, num=11)

gs2_coeffs = []
gas2_coeffs = []

for nbar in nbar_arr:
    fit_result = fit_NRF_polynomial_fixed_nbar(
        nbar=nbar,
        photon_cutoff=15,
        phonon_cutoff=15,
        t_final=1.0,
        gS_values=np.array([-0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.04]),
        gaS_values=np.array([-0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0, 0.03, 0.04]),
        print_tol=1e-7
    )
    gs2_coeffs.append(fit_result['coeffs'][3])
    gas2_coeffs.append(fit_result['coeffs'][4])
    print()
    print()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(nbar_arr, gs2_coeffs)
plt.plot(nbar_arr, gas2_coeffs)
plt.show()

In [ ]:
gs2_coeffs